# Лабораторная работа №11  
# Локальные малые модели и дистилляция

---

**Выполнил(а):** студент(ка) группы ______  
**ФИО:** _______________________________  
**Дата:** _______________________________

---

## 1. Цель работы

- Изучить особенности малых языковых моделей (SLM)
- Освоить техники дистилляции знаний
- Научиться запускать модели локально
- Сравнить качество SLM и больших моделей
- Оценить производительность на CPU/GPU

---

## 2. Теоретические сведения

### 2.1. Малые языковые модели (SLM)

| Модель | Параметры | Особенности |
|--------|-----------|-------------|
| TinyLlama | 1.1B | Быстрая, для edge devices |
| Phi-3 Mini | 3.8B | Высокое качество для размера |
| Gemma 2B | 2B | От Google, эффективная |
| Qwen2.5-0.5B | 0.5B | Ультра-малая модель |

### 2.2. Дистилляция знаний

**Дистилляция** — передача знаний от большой модели (teacher) к малой (student):
- **Hard labels** — обычные метки классов
- **Soft labels** — распределения вероятностей
- **Feature-based** — промежуточные представления

### 2.3. Преимущества SLM

- ✅ Быстрый инференс
- ✅ Меньше требований к памяти
- ✅ Запуск на CPU
- ✅ Ниже стоимость деплоя
- ❌ Меньшее качество на сложных задачах

---

## 3. Задание

### 3.1. Базовый уровень (обязательно)

1. Загрузите TinyLlama или Phi-3 Mini
2. Протестируйте на задачах генерации текста
3. Измерьте скорость генерации
4. Сравните с большой моделью
5. Проанализируйте компромиссы

### 3.2. Продвинутый уровень (дополнительно)

- Реализуйте простую дистилляцию
- Квантуйте модель до 4-bit
- Запустите на CPU и сравните скорость

### 3.3. Индивидуальное задание

Подберите SLM для конкретного use case:
- **Вариант A:** Чат-бот для мобильного приложения
- **Вариант B:** Классификатор текста для edge device

Обоснуйте выбор модели.

---

## 4. Ход работы

### 4.1. Подготовка окружения

In [ ]:
# Установка зависимостей
!pip install transformers torch accelerate -q

# Проверка
import transformers
import torch

print(f"Transformers version: {transformers.__version__}")
print(f"PyTorch version: {torch.__version__}")
print(f"GPU доступен: {torch.cuda.is_available()}")

### 4.2. Загрузка малой модели

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import time

# Выбор модели
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
# MODEL_NAME = "microsoft/phi-1_5"  # Альтернатива

print(f"Загрузка модели: {MODEL_NAME}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None
)

print("Модель загружена!")

# Подсчет параметров
total_params = sum(p.numel() for p in model.parameters())
print(f"Параметров: {total_params:,} ({total_params/1e9:.2f}B)")

### 4.3. Тестирование генерации

In [ ]:
def generate_with_timing(prompt, max_tokens=50):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    start = time.time()
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    elapsed = time.time() - start
    
    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    tokens_per_sec = max_tokens / elapsed if elapsed > 0 else 0
    
    return text, elapsed, tokens_per_sec

# Тестовые запросы
test_prompts = [
    "Расскажи о преимуществах Python",
    "Что такое машинное обучение?",
    "Объясни принцип работы нейронных сетей"
]

print("=== Тестирование генерации ===")
for prompt in test_prompts:
    print(f"\nЗапрос: {prompt}")
    response, elapsed, tps = generate_with_timing(prompt, max_tokens=30)
    print(f"Ответ: {response[:150]}...")
    print(f"Время: {elapsed:.2f} сек, Токенов/сек: {tps:.1f}")

### 4.4. Сравнение с большой моделью

In [ ]:
# Сравнение характеристик
comparison_data = {
    "TinyLlama": {"params": "1.1B", "speed": "~50 tok/s", "quality": "Базовый"},
    "Phi-3 Mini": {"params": "3.8B", "speed": "~30 tok/s", "quality": "Хороший"},
    "Llama-2-7B": {"params": "7B", "speed": "~15 tok/s", "quality": "Высокий"},
    "GPT-4": {"params": "~1T", "speed": "~10 tok/s", "quality": "Отличный"}
}

print("=== Сравнение моделей ===")
print(f"{'Модель':<15} {'Параметры':<12} {'Скорость':<15} {'Качество'}")
print("-" * 55)
for name, data in comparison_data.items():
    print(f"{name:<15} {data['params']:<12} {data['speed']:<15} {data['quality']}")

### 4.5. Запуск на CPU

In [ ]:
# Принудительный запуск на CPU
cpu_model = model.cpu()
print("Модель перемещена на CPU")

print("\n=== Тест на CPU ===")
prompt = "Какие бывают структуры данных?"
response, elapsed, tps = generate_with_timing(prompt, max_tokens=20)
print(f"Запрос: {prompt}")
print(f"Время: {elapsed:.2f} сек, Токенов/сек: {tps:.1f}")
print(f"Ответ: {response[:100]}...")

### 4.6. Основы дистилляции

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class KnowledgeDistillationLoss(nn.Module):
    """Loss функция для дистилляции знаний"""
    
    def __init__(self, temperature=4.0, alpha=0.5):
        super().__init__()
        self.temperature = temperature
        self.alpha = alpha
        self.ce_loss = nn.CrossEntropyLoss()
    
    def forward(self, student_logits, teacher_logits, labels):
        # Soft loss (distillation)
        soft_loss = F.kl_div(
            F.log_softmax(student_logits / self.temperature, dim=1),
            F.softmax(teacher_logits / self.temperature, dim=1),
            reduction='batchmean'
        ) * (self.temperature ** 2)
        
        # Hard loss (standard CE)
        hard_loss = self.ce_loss(student_logits, labels)
        
        # Combined loss
        return self.alpha * soft_loss + (1 - self.alpha) * hard_loss

print("=== Компоненты дистилляции ===")
print("1. Teacher модель (большая) генерирует soft labels")
print("2. Student модель (малая) обучается на soft + hard labels")
print("3. Temperature параметр контролирует 'мягкость' распределения")
print("\nПример использования:")
print("  loss_fn = KnowledgeDistillationLoss(temperature=4.0)")
print("  loss = loss_fn(student_out, teacher_out, labels)")

### 4.7. Рекомендации по выбору SLM

In [ ]:
selection_guide = """
=== Гид по выбору малой модели ===

MOBILE DEVICES (<1GB RAM):
  → TinyLlama (1.1B), Qwen2.5-0.5B
  → Квантование: 4-bit обязательо

EDGE DEVICES (2-4GB RAM):
  → Phi-3 Mini (3.8B), Gemma 2B
  → Квантование: 8-bit рекомендуется

SERVER CPU (8+GB RAM):
  → Llama-2-7B, Mistral-7B
  → Можно без квантования

USE CASES:
  • Чат-боты: Phi-3, Gemma
  • Классификация: TinyLlama
  • Summarization: Mistral
  • Code generation: StarCoder

OPTIMIZATION TIPS:
  1. Используйте ONNX Runtime для CPU
  2. Применяйте квантование (bitsandbytes)
  3. Кэшируйте частые запросы
  4. Батчите запросы при возможности
"""

print(selection_guide)

---

## 5. Контрольные вопросы

1. Какие преимущества у малых моделей перед большими?
2. Что такое дистилляция знаний и как она работает?
3. Когда стоит использовать SLM вместо больших моделей?
4. Как квантование влияет на качество SLM?
5. Какие метрики важны при выборе SLM для продакшена?

---

## 6. Выводы

В ходе работы были изучены малые языковые модели, протестирована TinyLlama и рассмотрены основы дистилляции знаний для эффективного деплоя.

---